# 🏢 TX Multifamily CRE Valuation
## **Fine tune Summary**
1. **Target → `log(Price Per Unit)`**: reduces log-std from 0.84 to 0.59
2. **Target Encoding** for Submarket + Zip (smoothed, computed on train-fold only)
3. **Cap Rate Combined**: Actual filled by Pro Forma, then median — 30% coverage → better signal
4. **Amenities parsed**: 14 binary flags + amenity count
5. **Renovation features**: `Is_Renovated`, `Years_Since_Reno`
6. **Parking Spaces** added
7. **5-Fold Time-Series CV** for robust, leakage-free evaluation
8. **LightGBM** with early stopping

In [ ]:
# Install dependencies (Colab)
# !pip install lightgbm -q

import pandas as pd
import numpy as np
import warnings
import matplotlib.pyplot as plt
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import lightgbm as lgb
warnings.filterwarnings('ignore')

print('Libraries loaded ✓')

Libraries loaded ✓


## Step 1 — Load & Filter Data

In [ ]:
FILEPATH = '/content/Merged_Costar_TX_MF_Data.csv'  # adjust path as needed

df = pd.read_csv(FILEPATH)
df['Sale Date'] = pd.to_datetime(df['Sale Date'], format='mixed')

df = df[df['Property Type'].str.startswith('Multifamily', na=False)]
df = df[df['Sale Type'].isin(['Investment', 'Investment or Owner User'])]

df.dropna(axis=1, how='all', inplace=True)

distressed_flags = [
    'REO Sale', 'Auction Sale', 'Estate/Probate Sale',
    'Partial Interest Transfer', 'Zero Cash Flow Investment',
    'Ground Lease (Leasehold)', 'Ground Lease (Leased Fee)', 'LIHTC'
]
mask = df['Sale Condition'].apply(
    lambda x: any(f in str(x) for f in distressed_flags) if pd.notna(x) else False
)
df = df[~mask].copy()
print(f"After arm's-length filter: {len(df)} rows")

After arm's-length filter: 921 rows


## Step 2 — Target Variable: `log(Price Per Unit)` ✅ KEY CHANGE

**Why?** Predicting `log(Sale Price)` forces the model to learn both *how many units* a property has and *what each unit is worth* simultaneously.  
Predicting **Price Per Unit** isolates the valuation problem and reduces log-scale std from **0.84 → 0.59**.

In [ ]:
df = df.dropna(subset=['Number Of Units'])
df = df[df['Number Of Units'] >= 10]

df['Price_Per_Unit'] = df['Sale Price'] / df['Number Of Units']
lo, hi = df['Price_Per_Unit'].quantile([0.01, 0.99])
df = df[(df['Price_Per_Unit'] >= lo) & (df['Price_Per_Unit'] <= hi)]
df['Log_PPU'] = np.log(df['Price_Per_Unit'])

print(f'Clean dataset: {len(df)} rows')
print(f'PPU range:     ${lo:,.0f} – ${hi:,.0f}')
print(f'Median PPU:    ${df["Price_Per_Unit"].median():,.0f}')
print(f'log(PPU) std:  {df["Log_PPU"].std():.3f}  (vs log(Sale Price) std = {np.log(df["Sale Price"]).std():.3f})')

Clean dataset: 898 rows
PPU range:     $26,826 – $353,799
Median PPU:    $141,854
log(PPU) std:  0.454  (vs log(Sale Price) std = 0.759)


## Step 3 — Feature Engineering

In [ ]:
df = df.sort_values('Sale Date').reset_index(drop=True)

# Temporal
df['Sale_Year']        = df['Sale Date'].dt.year
df['Sale_Quarter']     = df['Sale Date'].dt.quarter
df['Days_Since_Start'] = (df['Sale Date'] - df['Sale Date'].min()).dt.days

# Building
df['Building_Age']    = (df['Sale_Year'] - df['Year Built']).where(lambda x: x >= 0, np.nan)
df['Log_Building_SF'] = np.log1p(df['Building SF'])
df['Log_Avg_Unit_SF'] = np.log1p(df['Avg Unit SF'].fillna(df['Avg Unit SF'].median()))
df['Log_Num_Units']   = np.log(df['Number Of Units'])
df['Num_Floors']      = df['Number Of Floors'].fillna(df['Number Of Floors'].median())
df['FAR']             = df['Floor Area Ratio'].fillna(df['Floor Area Ratio'].median())

# Quality
df['Building Class'] = df.groupby('Property Zip Code')['Building Class']\
    .transform(lambda x: x.fillna(x.mode()[0] if x.notna().sum() > 0 else np.nan))
df['Building Class'] = df.groupby('Submarket Name')['Building Class']\
    .transform(lambda x: x.fillna(x.mode()[0] if x.notna().sum() > 0 else np.nan))
df['Building_Class_Num'] = df['Building Class'].map({'C':1,'B':2,'A':3}).fillna(2)
df['Star_Rating']        = df['Star Rating'].map({'2 Star':1,'3 Star':2,'4 Star':3,'5 Star':4}).fillna(2)

# Cap rate — Actual → Pro Forma fallback (train median filled inside CV)
df['Cap_Rate_Combined'] = df['Actual Cap Rate'].fillna(df['Pro Forma Cap Rate'])
df['Cap_Rate_Known']    = df['Cap_Rate_Combined'].notna().astype(int)
print(f'Cap rate coverage (actual or pro forma): {df["Cap_Rate_Known"].mean():.1%}')

# Financials
df['Pct_Leased']         = df['Percent Leased'].fillna(df['Percent Leased'].median())
df['Log_Assessed_Value'] = np.log1p(df['Assessed Value'].fillna(df['Assessed Value'].median()))

# Renovation [NEW]
df['Is_Renovated']     = df['Year Renovated'].notna().astype(int)
df['Years_Since_Reno'] = (df['Sale_Year'] - df['Year Renovated']).clip(lower=0)
df['Years_Since_Reno'] = df['Years_Since_Reno'].fillna(df['Building_Age'])

# Parking [NEW]
df['Parking_Spaces'] = df['Number Of Parking Spaces'].fillna(df['Number Of Parking Spaces'].median())
df['Parking_Ratio']  = df['Parking Ratio'].fillna(df['Parking Ratio'].median())

# Unit mix
for col in ['Number Of Studios Units','Number Of 1 Bedrooms Units',
            'Number Of 2 Bedrooms Units','Number Of 3 Bedrooms Units']:
    df[col] = df[col].fillna(0)
df['Pct_Studio'] = df['Number Of Studios Units']    / df['Number Of Units']
df['Pct_1BR']    = df['Number Of 1 Bedrooms Units'] / df['Number Of Units']
df['Pct_2BR']    = df['Number Of 2 Bedrooms Units'] / df['Number Of Units']
df['Pct_3BR']    = df['Number Of 3 Bedrooms Units'] / df['Number Of Units']

# Amenities [NEW]
AMENITY_MAP = {
    'has_pool':        ['Pool'],
    'has_gym':         ['Fitness Center', 'Gym'],
    'has_spa':         ['Spa'],
    'has_gated':       ['Gated', 'Controlled Access'],
    'has_concierge':   ['Concierge'],
    'has_business':    ['Business Center'],
    'has_tennis':      ['Tennis'],
    'has_pkg_service': ['Package Service'],
    'has_dog_park':    ['Dog Park', 'Pet'],
    'has_rooftop':     ['Rooftop'],
    'has_cowork':      ['Co-Working', 'CoWorking'],
    'has_storage':     ['Storage'],
    'has_ev_charger':  ['EV Charging', 'Electric Vehicle'],
    'has_valet':       ['Valet'],
}
amenity_col = df['Amenities'].fillna('')
for feat, keywords in AMENITY_MAP.items():
    df[feat] = amenity_col.apply(lambda x: int(any(k.lower() in x.lower() for k in keywords)))
df['Amenity_Count'] = df[[k for k in AMENITY_MAP]].sum(axis=1)

# Location
df['Is_CBD']         = (df['Location Type'] == 'CBD').astype(int)
df['Is_Suburban']    = (df['Location Type'] == 'Suburban').astype(int)
df['Submarket_Name'] = df['Submarket Name'].str.strip().str.title()
df['Zip5']           = df['Property Zip Code'].astype(str).str[:5]

# Land
land_cap = df['Land Area SF'].quantile(0.99)
df['Land_Area_SF'] = df['Land Area SF'].fillna(df['Land Area SF'].median()).clip(upper=land_cap)

print('Feature engineering complete ✓')

Cap rate coverage (actual or pro forma): 31.3%
Feature engineering complete ✓


## Step 4 — Target Encoding Helper ✅ KEY CHANGE

**Why not OHE?** The original notebook OneHotEncoded 206 submarkets and 545 zip codes — **750+ sparse features for ~700 training rows** — causing the curse of dimensionality.  
**Target Encoding** replaces each category with its smoothed mean `Log_PPU`, computed **on the training fold only** to prevent leakage.

In [ ]:
def target_encode(train_df, test_df, col, target, smoothing=10):
    global_mean = train_df[target].mean()
    stats = train_df.groupby(col)[target].agg(['mean', 'count'])
    stats['smooth'] = (
        (stats['mean'] * stats['count'] + global_mean * smoothing)
        / (stats['count'] + smoothing)
    )
    train_enc = train_df[col].map(stats['smooth']).fillna(global_mean)
    test_enc  = test_df[col].map(stats['smooth']).fillna(global_mean)
    return train_enc, test_enc

print('Target encoding helper defined ✓')

Target encoding helper defined ✓


## Step 5 — Feature Set & Model Parameters

In [ ]:
NUMERIC_FEATURES = [
    # Size & structure
    'Log_Building_SF', 'Log_Num_Units', 'Log_Avg_Unit_SF', 'Num_Floors', 'FAR', 'Land_Area_SF',
    # Quality
    'Building_Class_Num', 'Star_Rating', 'Building_Age',
    # Unit mix
    'Pct_Studio', 'Pct_1BR', 'Pct_2BR', 'Pct_3BR',
    # Income / financials
    'Cap_Rate_Combined', 'Cap_Rate_Known', 'Log_Assessed_Value', 'Pct_Leased',
    # Renovation & parking
    'Is_Renovated', 'Years_Since_Reno', 'Parking_Spaces', 'Parking_Ratio',
    # Amenities
    'Amenity_Count',
    'has_pool','has_gym','has_spa','has_gated','has_concierge',
    'has_business','has_pkg_service','has_dog_park','has_rooftop',
    # Location
    'Is_CBD', 'Is_Suburban',
    'Sub_TE', 'Zip_TE',   # target-encoded — computed inside CV loop
    # Timing
    'Sale_Year', 'Sale_Quarter', 'Days_Since_Start',
]

TARGET = 'Log_PPU'

# LightGBM parameters — tuned for small CRE datasets (~900 rows)
PARAMS = {
    'objective':        'regression',
    'metric':           'rmse',
    'learning_rate':    0.03,
    'num_leaves':       31,
    'min_data_in_leaf': 10,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq':     5,
    'reg_alpha':        0.1,
    'reg_lambda':       1.0,
    'verbose':          -1,
    'random_state':     42,
}

print(f'Total features: {len(NUMERIC_FEATURES)}  (vs original: 700+ from OHE)')


Total features: 38  (vs original: 700+ from OHE)
Columns with 100% missing values: []


## Step 6 — 5-Fold Time-Series Cross-Validation

In [ ]:
TSCV = TimeSeriesSplit(n_splits=5)
fold_results = []
oof_preds  = np.full(len(df), np.nan)
oof_actual = np.full(len(df), np.nan)

print('=== 5-Fold Time-Series Cross-Validation ===\n')

for fold, (tr_idx, te_idx) in enumerate(TSCV.split(df), 1):
    tr = df.iloc[tr_idx].copy()
    te = df.iloc[te_idx].copy()

    # Cap rate: fill with train median only (no leakage)
    cap_med = tr.loc[tr['Cap_Rate_Known']==1, 'Cap_Rate_Combined'].median()
    tr.loc[tr['Cap_Rate_Known']==0, 'Cap_Rate_Combined'] = cap_med
    te.loc[te['Cap_Rate_Known']==0, 'Cap_Rate_Combined'] = cap_med

    tr['Sub_TE'], te['Sub_TE'] = target_encode(tr, te, 'Submarket_Name', TARGET)
    tr['Zip_TE'],  te['Zip_TE'] = target_encode(tr, te, 'Zip5',           TARGET)

    X_tr = tr[NUMERIC_FEATURES].copy()
    X_te = te[NUMERIC_FEATURES].copy()
    for col in NUMERIC_FEATURES:
        med = X_tr[col].median()
        X_tr[col] = X_tr[col].fillna(med)
        X_te[col] = X_te[col].fillna(med)

    lgb_tr = lgb.Dataset(X_tr, label=tr[TARGET])
    lgb_va = lgb.Dataset(X_te, label=te[TARGET], reference=lgb_tr)
    cb = lgb.train(
        PARAMS, lgb_tr, num_boost_round=2000, valid_sets=[lgb_va],
        callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)]
    )

    pred_log           = cb.predict(X_te)
    oof_preds[te_idx]  = pred_log
    oof_actual[te_idx] = te[TARGET].values

    actual_ppu = np.exp(te[TARGET].values)
    pred_ppu   = np.exp(pred_log)
    r2   = r2_score(te[TARGET], pred_log)
    rmse = np.sqrt(mean_squared_error(te[TARGET], pred_log))
    mape = np.mean(np.abs((actual_ppu - pred_ppu) / actual_ppu)) * 100
    mae  = mean_absolute_error(actual_ppu, pred_ppu)

    fold_results.append({'Fold':fold, 'R2':r2, 'RMSE':rmse, 'MAE/unit':mae, 'MAPE':mape})
    print(f'Fold {fold} | N_train={len(tr):3d} | R²={r2:.3f}  RMSE={rmse:.3f}  '
          f'MAE/unit=${mae:,.0f}  MAPE={mape:.1f}%')

valid          = ~np.isnan(oof_preds)
oof_r2         = r2_score(oof_actual[valid], oof_preds[valid])
oof_rmse       = np.sqrt(mean_squared_error(oof_actual[valid], oof_preds[valid]))
oof_ppu_pred   = np.exp(oof_preds[valid])
oof_ppu_actual = np.exp(oof_actual[valid])
oof_mape       = np.mean(np.abs((oof_ppu_actual - oof_ppu_pred) / oof_ppu_actual)) * 100
oof_mae        = mean_absolute_error(oof_ppu_actual, oof_ppu_pred)

print(f'\n{"─"*70}')
print(f'OOF AGGREGATE  R²={oof_r2:.3f}  RMSE={oof_rmse:.3f}  '
      f'MAE/unit=${oof_mae:,.0f}  MAPE={oof_mape:.1f}%')

=== 5-Fold Time-Series Cross-Validation ===

Fold 1 | N_train=153 | R²=0.505  RMSE=0.273  MAE/unit=$24,278  MAPE=20.3%
Fold 2 | N_train=302 | R²=0.476  RMSE=0.323  MAE/unit=$34,384  MAPE=25.3%
Fold 3 | N_train=451 | R²=0.349  RMSE=0.340  MAE/unit=$45,885  MAPE=25.0%
Fold 4 | N_train=600 | R²=0.347  RMSE=0.390  MAE/unit=$36,963  MAPE=34.5%
Fold 5 | N_train=749 | R²=0.464  RMSE=0.356  MAE/unit=$33,995  MAPE=30.2%

──────────────────────────────────────────────────────────────────────
OOF AGGREGATE  R²=0.447  RMSE=0.339  MAE/unit=$35,101  MAPE=27.1%


# **## Step 7 — Final Holdout Evaluation (most recent 20%)**

In [ ]:
split80 = int(len(df) * 0.8)
df_tr80 = df.iloc[:split80].copy()
df_te20 = df.iloc[split80:].copy()

df_tr80['Sub_TE'], df_te20['Sub_TE'] = target_encode(df_tr80, df_te20, 'Submarket_Name', TARGET)
df_tr80['Zip_TE'],  df_te20['Zip_TE'] = target_encode(df_tr80, df_te20, 'Zip5',           TARGET)

cap_med80 = df_tr80.loc[df_tr80['Cap_Rate_Known']==1, 'Cap_Rate_Combined'].median()
df_tr80.loc[df_tr80['Cap_Rate_Known']==0, 'Cap_Rate_Combined'] = cap_med80
df_te20.loc[df_te20['Cap_Rate_Known']==0, 'Cap_Rate_Combined'] = cap_med80

def prep(subset, ref=None):
    X = subset[NUMERIC_FEATURES].copy()
    for col in NUMERIC_FEATURES:
        med = ref[col].median() if ref is not None else X[col].median()
        X[col] = X[col].fillna(med)
    return X

X_tr_f = prep(df_tr80)
X_te_f = prep(df_te20, ref=df_tr80[NUMERIC_FEATURES])
y_tr_f = df_tr80[TARGET]
y_te_f = df_te20[TARGET]

ds_tr = lgb.Dataset(X_tr_f, label=y_tr_f)
ds_va = lgb.Dataset(X_te_f, label=y_te_f, reference=ds_tr)
final_model = lgb.train(
    PARAMS, ds_tr, num_boost_round=2000, valid_sets=[ds_va],
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(100)]
)

pred_log_f   = final_model.predict(X_te_f)
actual_ppu_f = np.exp(y_te_f.values)
pred_ppu_f   = np.exp(pred_log_f)

r2_f   = r2_score(y_te_f, pred_log_f)
rmse_f = np.sqrt(mean_squared_error(y_te_f, pred_log_f))
mape_f = np.mean(np.abs((actual_ppu_f - pred_ppu_f) / actual_ppu_f)) * 100
mae_f  = mean_absolute_error(actual_ppu_f, pred_ppu_f)
mre_f  = np.mean((actual_ppu_f - pred_ppu_f) / actual_ppu_f) * 100
mae_total = mean_absolute_error(
    actual_ppu_f * df_te20['Number Of Units'].values,
    pred_ppu_f   * df_te20['Number Of Units'].values
)

print('=== Final Holdout Results (most recent 20% of data) ===')
print(f'  R²:               {r2_f:.3f}')
print(f'  RMSE (log scale): {rmse_f:.3f}')
print(f'  MAE / unit:       ${mae_f:,.0f}')
print(f'  MAPE:             {mape_f:.1f}%')
print(f'  MRE:              {mre_f:+.1f}%  (+ = under-predicts, - = over-predicts)')
print(f'  MAE total $:      ${mae_total:,.0f}')

[100]	valid_0's rmse: 0.345608
[200]	valid_0's rmse: 0.340962
=== Final Holdout Results (most recent 20% of data) ===
  R²:               0.469
  RMSE (log scale): 0.341
  MAE / unit:       $33,147
  MAPE:             26.9%
  MRE:              -5.4%  (+ = under-predicts, - = over-predicts)
  MAE total $:      $8,067,678


# **XGBoost and random forest**

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

def train_and_evaluate_model(model_name, model_instance, X_train, y_train, X_test, y_test, df_test_units):
    print(f'\n=== {model_name} Model Training and Evaluation ===')
    model_instance.fit(X_train, y_train)

    pred_log = model_instance.predict(X_test)
    actual_ppu = np.exp(y_test.values)
    pred_ppu = np.exp(pred_log)

    r2   = r2_score(y_test, pred_log)
    rmse = np.sqrt(mean_squared_error(y_test, pred_log))
    mape = np.mean(np.abs((actual_ppu - pred_ppu) / actual_ppu)) * 100
    mae  = mean_absolute_error(actual_ppu, pred_ppu)

    print(f'  R²:               {r2:.3f}')
    print(f'  RMSE (log scale): {rmse:.3f}')
    print(f'  MAE / unit:       ${mae:,.0f}')
    print(f'  MAPE:             {mape:.1f}%')

    return {'Model': model_name, 'R²': round(r2,3), 'RMSE': round(rmse,3), 'MAPE (%)': round(mape,1), 'Target': 'log(Price/Unit)', 'CV': 'Holdout 20%'}


In [ ]:
# Random Forest Model
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1, max_depth=10, min_samples_leaf=5)
rf_results = train_and_evaluate_model(
    'Random Forest (NEW)', rf_model, X_tr_f, y_tr_f, X_te_f, y_te_f, df_te20['Number Of Units']
)



=== Random Forest (NEW) Model Training and Evaluation ===
  R²:               0.354
  RMSE (log scale): 0.375
  MAE / unit:       $36,697
  MAPE:             30.3%


In [ ]:
# XGBoost Model
xgb_model = XGBRegressor(n_estimators=1000, learning_rate=0.03, max_depth=7, subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1, tree_method='hist')
xgb_results = train_and_evaluate_model(
    'XGBoost (NEW)', xgb_model, X_tr_f, y_tr_f, X_te_f, y_te_f, df_te20['Number Of Units']
)



=== XGBoost (NEW) Model Training and Evaluation ===
  R²:               0.364
  RMSE (log scale): 0.373
  MAE / unit:       $36,151
  MAPE:             29.5%


In [ ]:
# Update comparison DataFrame
comparison = pd.concat([
    comparison,
    pd.DataFrame([rf_results]),
    pd.DataFrame([xgb_results])
], ignore_index=True)
display(comparison)

NameError: name 'comparison' is not defined

## Step 8 — Visualisations

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

ax = axes[0]
ax.scatter(actual_ppu_f/1e3, pred_ppu_f/1e3, alpha=0.4, s=20, color='steelblue')
lims = [min(actual_ppu_f.min(), pred_ppu_f.min())/1e3,
        max(actual_ppu_f.max(), pred_ppu_f.max())/1e3]
ax.plot(lims, lims, 'r--', lw=1.5, label='Perfect')
ax.set_xlabel('Actual Price/Unit ($K)')
ax.set_ylabel('Predicted Price/Unit ($K)')
ax.set_title(f'Predicted vs Actual (Holdout)\nMAPE={mape_f:.1f}%  R²={r2_f:.3f}')
ax.legend(fontsize=9)

ax2 = axes[1]
folds = [r['Fold'] for r in fold_results]
mapes = [r['MAPE'] for r in fold_results]
ax2.bar(folds, mapes, color='steelblue', edgecolor='white')
ax2.axhline(np.mean(mapes), color='red', linestyle='--', lw=1.5,
            label=f'Mean={np.mean(mapes):.1f}%')
ax2.set_xlabel('CV Fold')
ax2.set_ylabel('MAPE (%)')
ax2.set_title('MAPE by CV Fold')
ax2.legend(fontsize=9)

ax3 = axes[2]
imp = pd.Series(
    final_model.feature_importance(importance_type='gain'),
    index=NUMERIC_FEATURES
).sort_values().tail(15)
imp.plot(kind='barh', ax=ax3, color='steelblue')
ax3.set_title('Feature Importance (Gain, Top 15)')
ax3.set_xlabel('Gain')

plt.tight_layout()
plt.show()

## Step 9 — Model Comparison

In [ ]:
comparison = pd.DataFrame([
    {'Model': 'XGBoost (original)',           'R²': 0.675, 'RMSE': 0.443, 'MAPE (%)': 36.4, 'Target': 'log(Sale Price)',  'CV': 'Single 80/20'},
    {'Model': 'Random Forest (original)',     'R²': 0.347, 'RMSE': 0.628, 'MAPE (%)': 53.9, 'Target': 'log(Sale Price)',  'CV': 'Single 80/20'},
    {'Model': 'Optimized XGBoost (original)', 'R²': 0.651, 'RMSE': 0.459, 'MAPE (%)': 35.3, 'Target': 'log(Sale Price)',  'CV': 'Single 80/20'},
    {'Model': 'LightGBM + TE (NEW, OOF)',     'R²': round(oof_r2,3), 'RMSE': round(oof_rmse,3), 'MAPE (%)': round(oof_mape,1), 'Target': 'log(Price/Unit)', 'CV': '5-Fold TS CV'},
    {'Model': 'LightGBM + TE (NEW, holdout)', 'R²': round(r2_f,3),  'RMSE': round(rmse_f,3),  'MAPE (%)': round(mape_f,1),  'Target': 'log(Price/Unit)', 'CV': 'Holdout 20%'},
])
display(comparison)

improvement = (36.4 - mape_f) / 36.4 * 100
print(f'\nMAPE improvement over best original model: {improvement:.0f}% reduction')

## Step 10 — Train Production Model & Score a New Listing

In [ ]:
# Train on ALL data using best_iteration from holdout run
cap_med_all = df.loc[df['Cap_Rate_Known']==1, 'Cap_Rate_Combined'].median()
df.loc[df['Cap_Rate_Known']==0, 'Cap_Rate_Combined'] = cap_med_all
df['Sub_TE'], _ = target_encode(df_tr80, df, 'Submarket_Name', TARGET)
df['Zip_TE'],  _ = target_encode(df_tr80, df, 'Zip5',           TARGET)

X_all = prep(df)
ds_all = lgb.Dataset(X_all, label=df[TARGET])
production_model = lgb.train(
    PARAMS, ds_all,
    num_boost_round=final_model.best_iteration + 50
)
print('Production model trained on all data ✓')

# ── Score a new listing ───────────────────────────────────────────────────
sub_global_mean = df[TARGET].mean()

new_listing = pd.DataFrame([{
    'Log_Building_SF':    np.log1p(150_000),
    'Log_Num_Units':      np.log(300),
    'Log_Avg_Unit_SF':    np.log1p(950),
    'Num_Floors':         4,
    'FAR':                1.2,
    'Land_Area_SF':       80_000,
    'Building_Class_Num': 2,
    'Star_Rating':        3,
    'Building_Age':       8,
    'Pct_Studio':         0.05,
    'Pct_1BR':            0.35,
    'Pct_2BR':            0.50,
    'Pct_3BR':            0.10,
    'Cap_Rate_Combined':  5.2,
    'Cap_Rate_Known':     1,
    'Log_Assessed_Value': np.log1p(38_000_000),
    'Pct_Leased':         93.0,
    'Is_Renovated':       0,
    'Years_Since_Reno':   8,
    'Parking_Spaces':     350,
    'Parking_Ratio':      1.17,
    'Amenity_Count':      5,
    'has_pool':1,'has_gym':1,'has_spa':0,'has_gated':1,'has_concierge':0,
    'has_business':1,'has_pkg_service':1,'has_dog_park':0,'has_rooftop':0,
    'Is_CBD':0,'Is_Suburban':1,
    'Sub_TE': sub_global_mean,
    'Zip_TE': sub_global_mean,
    'Sale_Year':      2026,
    'Sale_Quarter':   1,
    'Days_Since_Start': (pd.Timestamp('2026-01-15') - df['Sale Date'].min()).days,
}])

N_UNITS   = 300
ASK_PRICE = 52_000_000
MARGIN    = 0.05

pred_ppu   = np.exp(production_model.predict(new_listing[NUMERIC_FEATURES])[0])
pred_total = pred_ppu * N_UNITS
diff       = pred_total - ASK_PRICE
diff_pct   = diff / ASK_PRICE * 100

if pred_total > ASK_PRICE * (1 + MARGIN):
    signal, emoji = 'UNDERPRICED — BID', '🟢'
elif pred_total < ASK_PRICE * (1 - MARGIN):
    signal, emoji = 'OVERPRICED — PASS', '🔴'
else:
    signal, emoji = 'FAIR VALUE — NEGOTIATE', '🟡'

print(f'\n─── New Listing Valuation ─────────────────────────')
print(f'  Units:           {N_UNITS}')
print(f'  Asking Price:    ${ASK_PRICE:,.0f}')
print(f'  Model FMV/unit:  ${pred_ppu:,.0f}')
print(f'  Model FMV total: ${pred_total:,.0f}')
print(f'  Difference:      ${diff:+,.0f}  ({diff_pct:+.1f}%)')
print(f'  Signal:          {emoji}  {signal}')
print(f'───────────────────────────────────────────────────')